In [26]:
import pandas as pd
import cv2
import ast
import numpy as np
import random

def get_color_by_id(traj_id):
    """Gera uma cor consistente (BGR) baseada no ID da trajetória."""
    random.seed(traj_id)
    # Cores um pouco mais claras para o rastro ficar bonito
    return (random.randint(100, 255), random.randint(100, 255), random.randint(100, 255))

def interpolar_posicao(pontos, progresso_total):
    """Interpolação linear para movimento suave."""
    n_pontos = len(pontos)
    if n_pontos == 0: return 0, 0
    if n_pontos == 1: return pontos[0]
    
    posicao_float = progresso_total * (n_pontos - 1)
    idx_anterior = int(posicao_float)
    idx_proximo = min(idx_anterior + 1, n_pontos - 1)
    fator = posicao_float - idx_anterior
    
    p_ant = pontos[idx_anterior]
    p_prox = pontos[idx_proximo]
    
    x = p_ant[0] * (1 - fator) + p_prox[0] * fator
    y = p_ant[1] * (1 - fator) + p_prox[1] * fator
    return x, y

def gerar_video_com_rastro(video_input, csv_input, video_output, target_user_id, target_trajectory_id=None):
    # 1. Carregar Dados
    df = pd.read_csv(csv_input)
    df = df[df['user_id'] == target_user_id].copy()
    
    if target_trajectory_id is not None:
        df = df[df['trajectory_id'] == 'traj_'+ str(target_trajectory_id)]
        print(f"Filtrando apenas pela trajetória: {target_trajectory_id}")
    else:
        print(f"Processando TODAS as trajetórias do usuário {target_user_id} (Cores distintas)")

    if df.empty:
        print("Nenhuma trajetória encontrada com os filtros atuais.")
        return

    df['trajectory_xy'] = df['trajectory_xy'].apply(ast.literal_eval)

    # 2. Setup Vídeo
    cap = cv2.VideoCapture(video_input)
    if not cap.isOpened():
        print("Erro ao abrir vídeo.")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    out = cv2.VideoWriter(video_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # 3. Preparar Trajetórias e Histórico
    trajetorias = []
    for _, row in df.iterrows():
        trajetorias.append({
            'id': row['trajectory_id'],
            'inicio': int(row['frame_inicial']),
            'fim': int(row['frame_final']),
            'coords': row['trajectory_xy'],
            'color': get_color_by_id(row['trajectory_id']),
            'history_pixels': [] # Nova lista para guardar o rastro em pixels
        })

    # 4. Loop Frame a Frame
    frame_atual = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for traj in trajetorias:
            # Verifica se a trajetória já começou e ainda não terminou
            if traj['inicio'] <= frame_atual <= traj['fim']:
                duracao = traj['fim'] - traj['inicio']
                progresso = (frame_atual - traj['inicio']) / duracao if duracao > 0 else 0
                
                # Cálculo da posição atual
                x_raw, y_raw = interpolar_posicao(traj['coords'], progresso)
                
                # Projeção
                if x_raw <= 1.5 and y_raw <= 1.5:
                    center_x = int(x_raw * width)
                    center_y = int(y_raw * height)
                else:
                    center_x = int(x_raw)
                    center_y = int(y_raw)

                current_point = (center_x, center_y)

                # --- NOVO: Adicionar ao histórico e desenhar rastro ---
                # Adiciona o ponto atual ao histórico
                traj['history_pixels'].append(current_point)
                
                # Desenha o rastro se houver pelo menos 2 pontos
                if len(traj['history_pixels']) > 1:
                    # Converte para formato numpy necessário para polylines
                    pts = np.array(traj['history_pixels'], np.int32)
                    pts = pts.reshape((-1, 1, 2))
                    # Desenha a linha (rastro) com espessura menor que a caixa
                    cv2.polylines(frame, [pts], isClosed=False, color=traj['color'], thickness=2)
                # ------------------------------------------------------

                # Desenho da Bounding Box
                box_size = 50
                p1 = (center_x - box_size//2, center_y - box_size//2)
                p2 = (center_x + box_size//2, center_y + box_size//2)
                color = traj['color']
                
                cv2.rectangle(frame, p1, p2, color, 3)
                
                # Label
                label = str(traj['id'])
                (w_text, h_text), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (p1[0], p1[1]-25), (p1[0] + w_text, p1[1]), color, -1)
                cv2.putText(frame, label, (p1[0], p1[1]-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)

        out.write(frame)
        frame_atual += 1
        
        if frame_atual % 100 == 0:
            print(f"Processado: {frame_atual}/{total_frames}")

    cap.release()
    out.release()
    print("Concluído com rastro.")

# --- EXEMPLO DE USO ---
# Altere os nomes dos arquivos conforme necessário
gerar_video_com_rastro('38.mp4', 'symbolic.csv', 'saida_com_rastro.mp4', target_user_id=38, target_trajectory_id=12)

Filtrando apenas pela trajetória: 12
Processado: 100/1470
Processado: 200/1470
Processado: 300/1470
Processado: 400/1470
Processado: 500/1470
Processado: 600/1470
Processado: 700/1470
Processado: 800/1470
Processado: 900/1470
Processado: 1000/1470
Processado: 1100/1470
Processado: 1200/1470
Processado: 1300/1470
Processado: 1400/1470
Concluído com rastro.


In [24]:
df = pd.read_csv('symbolic.csv')
df = df[(df['user_id'] == 38) & (df['trajectory_id'] == 'traj_12')]
df

,Unnamed: 0,trajectory_id,user_id,frame_inicial,frame_final,symbolic_movement_10,symbolic_movement_15,symbolic_movement_20,movement_list,cluster,...,isomap_2,mds_1,mds_2,nmf_1,nmf_2,umap_1,umap_2,trajectory_xy,trajectory_xy_translate,trajectory_xy_rotated
11,11,traj_12,38,0,162,"['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...",1,...,-0.307369,0.580974,0.029432,0.119646,0.0,-21.844568,3.292662,"[(0.55703125, 0.4916666666666666), (0.64765625...","[(0.0, 0.0), (0.090625, 0.1097222222222223), (...","[(0.0, 0.0), (0.1420705002522309, 0.0082358747..."
